In [83]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [84]:
df = pd.read_csv(r"C:\Users\LENOVO\Desktop\fraud-detection-system\data\PS_20174392719_1491204439457_log.csv")

print(df.head())
print("Shape:", df.shape)

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
Shape: (6362620, 11)


In [85]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [86]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [87]:
print(df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [88]:
print(df["isFraud"].value_counts(normalize=True) * 100)

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [89]:
print(df["type"].value_counts())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [90]:
X = df[features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 5)
y shape: (6362620,)


In [91]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (5090096, 5)
Testing : (1272524, 5)


In [92]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

target = "isFraud"

X = df[features]
y = df[target]

print("Features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']

X shape: (6362620, 5)
y shape: (6362620,)


In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (5090096, 5)
Testing : (1272524, 5)


In [94]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["type"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessor created!")

Preprocessor created!


In [95]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

print("Scale Pos Weight:", scale_pos_weight)

Scale Pos Weight: 773.7482496194825


In [96]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline created!")

Pipeline created!


In [97]:
pipeline.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [98]:
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated!")
print(y_prob[:10])

Predictions generated!
[8.5284910e-06 9.5584440e-08 1.1975440e-05 3.5514157e-07 2.1119309e-05
 2.3735234e-05 1.7152520e-02 3.6879388e-05 1.6400336e-06 8.0595093e-08]


In [99]:
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.21      0.99      0.34      1643

    accuracy                           1.00   1272524
   macro avg       0.60      0.99      0.67   1272524
weighted avg       1.00      1.00      1.00   1272524



In [100]:
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.9995280663927968


In [101]:
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print("Training:", X_train2.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (4072076, 5)
Validation: (1018020, 5)
Test: (1272524, 5)


In [102]:
pipeline_val = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

print("Validation pipeline created!")

Validation pipeline created!


In [103]:
pipeline_val.fit(X_train2, y_train2)

print("Validation model training completed!")

Validation model training completed!


In [104]:
y_val_prob = pipeline_val.predict_proba(X_val)[:, 1]

print(y_val_prob[:10])

[8.81966997e-08 1.01265614e-04 4.34200956e-06 4.17112638e-08
 5.12399993e-06 1.07429441e-04 1.79682203e-07 1.40712325e-06
 3.83390798e-05 1.62977383e-08]


In [105]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_prob
)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

print("Best Threshold:", best_threshold)
print("Validation Precision:", precision[best_index])
print("Validation Recall:", recall[best_index])
print("Validation F1:", f1_scores[best_index])

Best Threshold: 0.9891073
Validation Precision: 0.8791946308724832
Validation Recall: 0.7975646879756468
Validation F1: 0.8363926575718265


In [106]:
y_test_prob = pipeline_val.predict_proba(X_test)[:, 1]

y_test_pred = (
    y_test_prob >= best_threshold
).astype(int)

print(classification_report(y_test, y_test_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.90      0.81      0.85      1643

    accuracy                           1.00   1272524
   macro avg       0.95      0.90      0.93   1272524
weighted avg       1.00      1.00      1.00   1272524

ROC-AUC: 0.9993880589924353


In [107]:
cm = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1270734     147]
 [    314    1329]]


In [108]:
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

print("Final pipeline created!")

Final pipeline created!


In [109]:
final_pipeline.fit(X_train, y_train)

print("Final model trained!")

Final model trained!


In [110]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_pipeline,
    "../models/fraud_paysim_pipeline.pkl"
)

joblib.dump(
    best_threshold,
    "../models/fraud_paysim_threshold.pkl"
)

print("Model saved successfully!")

Model saved successfully!
